# Diseño y preparación de la base de datos y Data Mart

Este notebook documenta el proceso para pasar del dataset procesado a una base de datos relacional y a un Data Mart dimensional. Se alinea con las indicaciones del Proyecto Integrador: documentar la limpieza y el ETL, y construir un Data Mart con tabla de hechos, dimensiones, llaves y medidas.

**Motor previsto:** Microsoft SQL Server, sujeto a confirmar en el equipo donde se ejecutará el proyecto.

**Fuente de carga:** `data/processed/clean/online_shoppers_intention_clean.csv`. El archivo original en `data/raw/` se conserva sin cambios.

**Procedimiento de carga:** Los datos se cargan a SQL Server mediante Python (`src/etl/cargar_sql.py`) usando consultas parametrizadas, no mediante `BULK INSERT` ni SQL puro. El script SQL `sql/dml/02_cargar_dataset.sql` ahora solo valida la carga.

## 1. Estado y objetivo

El dataset procesado disponible contiene 12,205 sesiones y 18 variables. En la revisión local se observaron cero duplicados exactos y cero valores faltantes. La variable `Month` está estandarizada con abreviaturas; por ejemplo, `June` se transformó en `Jun`.

El objetivo de esta etapa es almacenar las sesiones y sus atributos para consultar el comportamiento de compra. La base tendrá una zona de staging para recibir el CSV y un esquema estrella para el Data Mart.

In [3]:
from pathlib import Path
import pandas as pd

# Detectar raíz del proyecto de forma robusta
raiz = Path.cwd()
while raiz.name and raiz.name != 'REC_ECDB_2026_ChavezMolina_Byron':
    if raiz.parent == raiz:
        break
    raiz = raiz.parent

ruta_csv = raiz / 'data' / 'processed' / 'clean' / 'online_shoppers_intention_clean.csv'
datos = pd.read_csv(ruta_csv)

print(f'Filas: {datos.shape[0]:,}')
print(f'Columnas: {datos.shape[1]}')
print(f'Duplicados exactos: {datos.duplicated().sum():,}')
print(f'Valores faltantes: {datos.isna().sum().sum():,}')
print('Columnas:', datos.columns.tolist())

Filas: 12,205
Columnas: 18
Duplicados exactos: 0
Valores faltantes: 0
Columnas: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']


## 2. Grano y reglas del modelo

**Grano de `FactSesion`: una fila por cada sesión del dataset procesado.** El archivo fuente no incluye un identificador único de sesión, por lo que SQL Server generará una llave sustituta (`SesionKey`) para identificar cada fila dentro del Data Mart. Esa llave no representa un identificador original del sitio web.

Las variables categóricas se separan en dimensiones para facilitar filtros y agrupaciones. Algunas se almacenan como números en el CSV, pero representan códigos, no cantidades. Las variables de páginas, duración y tasas se conservan como medidas. `Revenue` se conserva como indicador de compra y variable objetivo para clasificación; `Weekend` se conserva como indicador de sesión.

## 3. Definición del grano y diseño dimensional del Data Mart

### 3.1 Qué significa el grano

El **grano** (o *grain*) de una tabla de hechos define **qué representa una sola fila** en esa tabla. En este Data Mart, el grano de `FactSesion` es:

> **Una fila por cada sesión de navegación de un visitante en el sitio web.**

Esto significa que cada registro en `FactSesion` corresponde a una visita individual de un usuario al comercio electrónico, independientemente de si esa visita terminó en compra (`Revenue = 1`) o no (`Revenue = 0`). El dataset original contiene 12,205 filas tras la limpieza, cada una representando una sesión única; por tanto, `FactSesion` tendrá exactamente 12,205 filas tras la carga.

### 3.2 Por qué se eligió este nivel de detalle

El análisis del comportamiento de compradores en comercio electrónico requiere entender **qué ocurre durante una visita concreta**:

- **Navegación:** número y tipo de páginas visitadas (administrativas, informativas, de producto).
- **Tiempo:** duración en cada tipo de página.
- **Calidad de la sesión:** tasas de rebote (`BounceRates`), salida (`ExitRates`), valor de página (`PageValues`).
- **Contexto temporal:** mes (`Month`), día de fin de semana (`Weekend`), proximidad a día especial (`SpecialDay`).
- **Entorno técnico:** sistema operativo, navegador, región, tipo de tráfico.
- **Perfil del visitante:** nuevo, recurrente u otro (`VisitorType`).
- **Resultado:** ¿hubo compra? (`Revenue`).

Una granularidad más fina (p. ej., a nivel de *pageview* o *click*) no existe en el dataset original. Una granularidad más gruesa (p. ej., agregado por día o por visitante) perdería la capacidad de comparar sesiones individuales con y sin compra, que es el objetivo central del modelado supervisado (clasificación) y del análisis no supervisado (clustering).

### 3.3 Qué representa una fila de `FactSesion`

| Componente | Descripción |
|------------|-------------|
| **Llaves foráneas** (6) | Conectan la sesión con sus dimensiones: `MesKey`, `TipoVisitanteKey`, `SistemaOperativoKey`, `NavegadorKey`, `RegionKey`, `TipoTraficoKey`. |
| **Medidas de actividad** (6) | `Administrative`, `Administrative_Duration`, `Informational`, `Informational_Duration`, `ProductRelated`, `ProductRelated_Duration`. |
| **Tasas y valor** (3) | `BounceRates`, `ExitRates`, `PageValues`. |
| **Indicadores contextuales** (2) | `SpecialDay` (proximidad a día festivo), `Weekend` (sesión en fin de semana). |
| **Variable objetivo** | `Revenue` (1 = compra, 0 = no compra). |

### 3.4 Qué representa la clave primaria `SesionKey`

`SesionKey` es una **llave sustituta** (*surrogate key*) de tipo `BIGINT IDENTITY(1,1)` generada automáticamente por SQL Server al insertar cada fila en `FactSesion`.

- **No** proviene del dataset original (el CSV no tiene identificador de sesión).
- **No** tiene significado de negocio; solo sirve para identificar unívocamente la fila dentro del Data Mart.
- Permite referenciar la fact desde otras tablas si fuera necesario (p. ej., una tabla de hechos de detalle de clics futura).

### 3.5 Relación de cada sesión con las seis dimensiones

Cada sesión se relaciona **exactamente con una** fila de cada dimensión:

| Dimensión | Columna natural en CSV | Llave en `FactSesion` | Cardinalidad |
|-----------|------------------------|------------------------|--------------|
| `DimMes` | `Month` (ej. `Feb`, `Mar`) | `MesKey` | N:1 |
| `DimTipoVisitante` | `VisitorType` (ej. `Returning_Visitor`) | `TipoVisitanteKey` | N:1 |
| `DimSistemaOperativo` | `OperatingSystems` (código 1–8) | `SistemaOperativoKey` | N:1 |
| `DimNavegador` | `Browser` (código 1–13) | `NavegadorKey` | N:1 |
| `DimRegion` | `Region` (código 1–9) | `RegionKey` | N:1 |
| `DimTipoTrafico` | `TrafficType` (código 1–20) | `TipoTraficoKey` | N:1 |

Todas las dimensiones usan **llaves sustitutas enteras** como PK y tienen restricción `UNIQUE` sobre su atributo natural, lo que garantiza que no haya duplicados y soporta el `MERGE` upsert en la carga.

### 3.6 Medidas y atributos almacenados en la tabla de hechos

```sql
CREATE TABLE dbo.FactSesion (
    SesionKey BIGINT IDENTITY(1,1) PRIMARY KEY,
    -- Llaves foráneas (6 dimensiones)
    MesKey INT NOT NULL,
    TipoVisitanteKey INT NOT NULL,
    SistemaOperativoKey INT NOT NULL,
    NavegadorKey INT NOT NULL,
    RegionKey INT NOT NULL,
    TipoTraficoKey INT NOT NULL,
    -- Medidas de navegación
    Administrative INT NOT NULL,
    Administrative_Duration DECIMAL(14,6) NOT NULL,
    Informational INT NOT NULL,
    Informational_Duration DECIMAL(14,6) NOT NULL,
    ProductRelated INT NOT NULL,
    ProductRelated_Duration DECIMAL(14,6) NOT NULL,
    -- Tasas y valor
    BounceRates DECIMAL(12,8) NOT NULL,
    ExitRates DECIMAL(12,8) NOT NULL,
    PageValues DECIMAL(14,6) NOT NULL,
    -- Indicadores
    SpecialDay DECIMAL(8,6) NOT NULL,
    Weekend BIT NOT NULL,      -- 1 = fin de semana, 0 = entre semana
    Revenue BIT NOT NULL       -- 1 = compra, 0 = no compra
);
```

Todas las columnas son `NOT NULL` porque el dataset limpio no tiene faltantes y el ETL valida la integridad antes de cargar.

### 3.7 Cómo se identifica una compra mediante `Revenue`

`Revenue` es un indicador binario (`BIT` en SQL Server, `0`/`1` en Python):

- `Revenue = 1` → la sesión **terminó en compra** (el visitante realizó al menos una transacción).
- `Revenue = 0` → la sesión **no terminó en compra** (navegación sin transacción).

Es la **variable objetivo** para:
- **Clasificación supervisada:** predecir si una sesión nueva comprará.
- **Análisis descriptivo:** comparar comportamiento (páginas, tiempo, tasas) entre compradores y no compradores.
- **Segmentación no supervisada:** los clusters de K-Means pueden interpretarse según su tasa de `Revenue`.

### 3.8 Justificación del diseño dimensional

#### Análisis a nivel de sesión

El grano de sesión permite:

1. **Analizar comportamiento de navegación** sesión a sesión: cuántas páginas de cada tipo, cuánto tiempo, qué tasas de rebote/salida.
2. **Comparar sesiones con compra vs. sin compra** directamente, fila a fila, sin agregaciones que oculten variabilidad.
3. **Estudiar el comportamiento por contexto** usando las dimensiones: ¿compran más los `Returning_Visitor` en `Nov`? ¿Qué `TrafficType` trae sesiones con mayor `PageValues`?
4. **Alimentar modelos de Machine Learning** tal como vienen del dataset original:
   - **Regresión:** `ProductRelated_Duration` como target, features de la misma fila.
   - **Clasificación:** `Revenue` como target, resto de columnas como features.
   - **Clustering:** todas las variables numéricas de la fila para agrupar sesiones similares.

#### Esquema estrella (relaciones)

```mermaid
erDiagram
    DimMes ||--o{ FactSesion : "MesKey"
    DimTipoVisitante ||--o{ FactSesion : "TipoVisitanteKey"
    DimSistemaOperativo ||--o{ FactSesion : "SistemaOperativoKey"
    DimNavegador ||--o{ FactSesion : "NavegadorKey"
    DimRegion ||--o{ FactSesion : "RegionKey"
    DimTipoTrafico ||--o{ FactSesion : "TipoTraficoKey"

    FactSesion {
        BIGINT SesionKey PK
        INT MesKey FK
        INT TipoVisitanteKey FK
        INT SistemaOperativoKey FK
        INT NavegadorKey FK
        INT RegionKey FK
        INT TipoTraficoKey FK
        INT Administrative
        DECIMAL Administrative_Duration
        INT Informational
        DECIMAL Informational_Duration
        INT ProductRelated
        DECIMAL ProductRelated_Duration
        DECIMAL BounceRates
        DECIMAL ExitRates
        DECIMAL PageValues
        DECIMAL SpecialDay
        BIT Weekend
        BIT Revenue
    }

    DimMes {
        INT MesKey PK
        NVARCHAR(3) Mes UK
    }
    DimTipoVisitante {
        INT TipoVisitanteKey PK
        NVARCHAR(30) TipoVisitante UK
    }
    DimSistemaOperativo {
        INT SistemaOperativoKey PK
        INT CodigoSistemaOperativo UK
    }
    DimNavegador {
        INT NavegadorKey PK
        INT CodigoNavegador UK
    }
    DimRegion {
        INT RegionKey PK
        INT CodigoRegion UK
    }
    DimTipoTrafico {
        INT TipoTraficoKey PK
        INT CodigoTipoTrafico UK
    }
```

#### Verificación contra DDL y CSV

| Comprobación | Resultado |
|--------------|-----------|
| `FactSesion` tiene 1 fila por sesión del CSV (12,205) | ✅ El ETL inserta una fila por `iterrows()` del DataFrame limpio. |
| `SesionKey` es llave sustituta `IDENTITY`, no viene del CSV | ✅ DDL: `SesionKey BIGINT IDENTITY(1,1) PRIMARY KEY`. |
| 6 dimensiones con PK sustituta y UK en atributo natural | ✅ DDL: cada dimensión tiene `*Key IDENTITY PK` y `UNIQUE` en columna natural. |
| FK en `FactSesion` referencian PK de dimensiones | ✅ DDL: 6 `FOREIGN KEY` constraints declaradas. |
| `Weekend` y `Revenue` como `BIT` (0/1) en hecho, `NVARCHAR` en staging | ✅ Staging guarda 'True'/'False'; ETL convierte con `validar_bool_str()`. |
| `BounceRates` como `NVARCHAR` en staging, `DECIMAL` en hecho | ✅ Staging: `NVARCHAR(32)`; Hecho: `DECIMAL(12,8)`; ETL castea a `float`. |
| Medidas numéricas conservan precisión del CSV | ✅ `DECIMAL(14,6)` para duraciones, `DECIMAL(12,8)` para tasas. |
| Carga idempotente (re-ejecutable sin duplicados) | ✅ `TRUNCATE` staging/hecho + `MERGE` en dimensiones. |

> **Conclusión:** La implementación respeta estrictamente el grano definido. Cada sesión del CSV limpio genera una y solo una fila en `FactSesion`, vinculada a sus seis dimensiones mediante llaves sustitutas, con todas las medidas e indicadores necesarios para el análisis descriptivo, predictivo y de segmentación.

## 3. Mapeo de variables a tablas

| Tabla | Variables | Uso |
|---|---|---|
| `StgSesiones` | Las 18 columnas del CSV | Recibir la carga procesada y conservar una fila fuente por sesión. |
| `DimMes` | `Month` | Filtrar y agrupar por mes. |
| `DimTipoVisitante` | `VisitorType` | Analizar visitantes nuevos, recurrentes y otros. |
| `DimSistemaOperativo` | `OperatingSystems` | Analizar el código de sistema operativo. |
| `DimNavegador` | `Browser` | Analizar el código de navegador. |
| `DimRegion` | `Region` | Analizar el código de región. |
| `DimTipoTrafico` | `TrafficType` | Analizar el código de tipo de tráfico. |
| `FactSesion` | Llaves a dimensiones; cantidades de páginas; duraciones; tasas; `PageValues`; `SpecialDay`; `Weekend`; `Revenue` | Mantener una fila por sesión y sus medidas/indicadores. |

Las dimensiones tendrán llaves sustitutas enteras como PK. `FactSesion` guardará esas llaves como FK y tendrá su propia llave sustituta (`SesionKey`) generada por SQL Server. La carga reconstruye la tabla de hechos a partir de staging en cada ejecución.

## 4. Orden del proceso

1. Crear la base `ECBD_Compradores` en SQL Server.
2. Crear `StgSesiones` con las 18 variables del CSV, conservando una fila por sesión.
3. Ejecutar el ETL de Python que lee el CSV, limpia los datos y los carga a SQL Server:
   - `python -m src.etl.etl --cargar-sql`
4. Validar la carga con el script `sql/dml/02_cargar_dataset.sql` (solo validación).

### Paso 1: crear la base de datos

Ejecutar en una consulta de SQL Server Management Studio conectada al servidor. `USE` selecciona la base para las instrucciones posteriores.

```sql
IF DB_ID(N'ECBD_Compradores') IS NULL
BEGIN
    CREATE DATABASE ECBD_Compradores;
END;
GO

USE ECBD_Compradores;
GO

SELECT DB_NAME() AS BaseDeDatosActual;
```

La creación de la base de datos, la tabla de staging, las dimensiones y la tabla de hechos se encuentra implementada en el script `sql/ddl/01_crear_base_y_datamart.sql`.

La carga del dataset procesado se realiza desde Python ejecutando:
```powershell
.venv\Scripts\python.exe -m src.etl.etl --cargar-sql
```

La validación de la carga se realiza con `sql/dml/02_cargar_dataset.sql`, que verifica conteos, integridad referencial y muestra una muestra de datos.

## 5. Validaciones de carga previstas

Al terminar el ETL, comparar el número de filas de staging y de `FactSesion`. Como el dataset procesado tiene una fila por sesión y las dimensiones se construyen con valores distintos, la carga del hecho debe conservar las 12,205 sesiones. También se comprobará que no haya llaves foráneas sin correspondencia y que las dimensiones no tengan duplicados en sus atributos naturales.

Consultas de control incluidas en el script `sql/dml/02_cargar_dataset.sql` para revisar los registros cargados:

```sql
SELECT COUNT(*) AS FilasStaging FROM dbo.StgSesiones;
SELECT COUNT(*) AS FilasHechos FROM dbo.FactSesion;
SELECT TOP (5) * FROM dbo.FactSesion;
```

## 6. Estado de esta etapa

- Dataset procesado disponible: **sí**.
- Diseño conceptual y grano del hecho documentados: **sí**.
- Creación de la base, staging, dimensiones y tabla de hechos: **implementada en scripts SQL; pendiente de ejecución**.
- Carga del dataset y proceso ETL: **implementados en scripts SQL; pendientes de ejecución y validación**.
- Diagrama del Data Mart: **elaborado**.
- Validación de cantidades de registros y relaciones: **pendiente de comprobar en SQL Server**.

Este notebook documenta el diseño y permite revisar el dataset procesado con Python. La ejecución de los scripts SQL se realiza en SQL Server Management Studio.